# Notebook 03 — Modelo Predictivo Base (XGBoost + Calibración + SHAP)

Modelo de riesgo de no-asistencia con variables pre-tratamiento únicamente. **`SMS_received` queda excluido**: es la variable de tratamiento, se estima por separado en el Notebook 04 (IPW).

**Orden explícito (anti-fuga temporal):**
1. Test cronológico (último 20% de fechas) reservado desde el inicio.
2. Tuning de hiperparámetros con 3 folds walk-forward dentro del 80% inicial del periodo de entrenamiento (sin solaparse con `train_calib`).
3. Split cronológico del entrenamiento: `train_xgb` (primer 80%) + `train_calib` (último 20%).
4. Ajuste final de XGBoost en `train_xgb` con early stopping sobre `train_calib`.
5. Ajuste de calibración (Platt + Isotonic) sobre `train_calib`; selección por menor Brier con tiebreak ε=0.001 a Platt.
6. Evaluación final una sola vez en el test cronológico.

Toda la lógica vive en `src/modelado.py`; aquí orquestamos y narramos. Las decisiones de configuración (calibrador, grid de hiperparámetros, esquema de validación) se justifican en su sección correspondiente.

In [1]:
%load_ext autoreload
%autoreload 2

import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import modelado, rutas

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
SEED = rutas.SEED
np.random.seed(SEED)
modelado.configurar_estilo()
print(f"Seed fijado: {SEED}")

Seed fijado: 42


## 1. Carga y verificación

Se cargan los datos preprocesados de NB01 (`train_v1.csv`, `test_v1.csv`). La codificación de `Showed_up` se re-verifica (verificación obligatoria de codificación) antes de invertir el signo para construir el target.

In [2]:
train_df, test_df = modelado.cargar_train_test_nb03(rutas.CSV_TRAIN, rutas.CSV_TEST)
modelado.verificar_codificacion_showed_up_pred(train_df)
modelado.verificar_codificacion_showed_up_pred(test_df)
print(f"train: {train_df.shape}, test: {test_df.shape}")
print(f"train AppointmentDay: {train_df['AppointmentDay'].min().date()} → {train_df['AppointmentDay'].max().date()}")
print(f"test  AppointmentDay: {test_df['AppointmentDay'].min().date()} → {test_df['AppointmentDay'].max().date()}")

INFO Train cargado: 85657 filas, Test: 21330 filas. Train spans 2016-04-29 → 2016-06-01.


INFO Showed_up verificado: 79.3% asistencia. Target del modelo = 1 - Showed_up (no-show).


INFO Showed_up verificado: 81.4% asistencia. Target del modelo = 1 - Showed_up (no-show).


train: (85657, 28), test: (21330, 28)
train AppointmentDay: 2016-04-29 → 2016-06-01
test  AppointmentDay: 2016-06-02 → 2016-06-08


## 2. Construcción de la matriz de features y target

**Target:** `y = 1 - Showed_up` (positivo = no asistió). Facilita la interpretación de SHAP (signos positivos suben el riesgo de no-show) y de `scale_pos_weight` (la clase minoritaria coincide con el evento a predecir).

**Codificación de `Gender`:** `gender_F = 1` si F, `0` si M. Dirección documentada para SHAP.

**`prior_noshow_rate` con ~72% NaN:** XGBoost gestiona los NaN nativamente. No se imputa porque `NaN` aquí significa "sin historial observado dentro de la ventana del dataset" (censura por la izquierda) — NO "cero inasistencias previas". Esto se refleja también en la interpretación de SHAP: cuando `is_first_visit = 1`, ambas variables codifican la misma señal y SHAP repartirá atribución entre ellas.

In [3]:
X_train_full, y_train_full = modelado.preparar_features(train_df)
X_test, y_test = modelado.preparar_features(test_df)
print(f"Features ({len(modelado.FEATURES_NB03)}): {modelado.FEATURES_NB03}")
print(f"\nX_train_full: {X_train_full.shape}, prevalencia no-show train: {y_train_full.mean():.3f}")
print(f"X_test:       {X_test.shape}, prevalencia no-show test:  {y_test.mean():.3f}")
print(f"\nNaN por columna (train, sólo columnas con NaN):")
print(X_train_full.isna().sum()[X_train_full.isna().sum() > 0])

INFO X: (85657, 18), prevalencia no-show: 20.7%


INFO X: (21330, 18), prevalencia no-show: 18.6%


Features (18): ['lead_time', 'Age', 'gender_F', 'neighbourhood_encoded', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'comorbidity_count', 'chronic_flag', 'scheduled_weekday', 'scheduled_month', 'appointment_weekday', 'appointment_month', 'prior_appointment_count', 'prior_noshow_rate', 'is_first_visit']

X_train_full: (85657, 18), prevalencia no-show train: 0.207
X_test:       (21330, 18), prevalencia no-show test:  0.186

NaN por columna (train, sólo columnas con NaN):
prior_noshow_rate    65453
dtype: int64


## 3. Split cronológico `train_xgb` / `train_calib`

El periodo de entrenamiento se divide cronológicamente: primer 80% → `train_xgb` (ajuste del XGBoost), último 20% → `train_calib` (early stopping + ajuste del calibrador Platt/Isotonic).

In [4]:
X_xgb, y_xgb, X_calib, y_calib, fecha_corte_xgb_calib = modelado.dividir_train_xgb_calib(
    X_train_full, y_train_full, train_df["AppointmentDay"], frac_xgb=modelado.FRAC_TRAIN_XGB,
)
print(f"Corte cronológico: {fecha_corte_xgb_calib.date()}")
print(f"train_xgb:  {X_xgb.shape}, prev no-show: {y_xgb.mean():.3f}")
print(f"train_calib: {X_calib.shape}, prev no-show: {y_calib.mean():.3f}")

INFO Train→ train_xgb (≤ 2016-05-24): 69020 filas; train_calib (> 2016-05-24): 16637 filas.


Corte cronológico: 2016-05-24
train_xgb:  (69020, 18), prev no-show: 0.211
train_calib: (16637, 18), prev no-show: 0.189


## 4. Walk-forward CV: 3 folds expanding-window dentro de `train_xgb`

Los folds operan sobre `train_xgb` (no sobre el train completo), para evitar solapamiento entre las ventanas de validación de CV y el conjunto de calibración. El último 20% (`train_calib`) queda completamente fuera del tuning.

Los cortes se definen por cuantiles uniformes de `AppointmentDay` dentro de `train_xgb`:

| Fold | Train | Val |
|---|---|---|
| 1 | ≤ q50 | (q50, q67] |
| 2 | ≤ q67 | (q67, q83] |
| 3 | ≤ q83 | (q83, q100] |

In [5]:
folds = modelado.generar_folds_walk_forward(
    train_df["AppointmentDay"].loc[X_xgb.index], n_folds=modelado.N_FOLDS_WALK_FORWARD,
)
for f in folds:
    print(f.resumen())

INFO Fold 1: {'fold': 1, 'train_hasta': '2016-05-11', 'val_hasta': '2016-05-16', 'n_train': 36275, 'n_val': 12606}


INFO Fold 2: {'fold': 2, 'train_hasta': '2016-05-16', 'val_hasta': '2016-05-19', 'n_train': 48881, 'n_val': 12556}


INFO Fold 3: {'fold': 3, 'train_hasta': '2016-05-19', 'val_hasta': '2016-05-24', 'n_train': 61437, 'n_val': 7583}


{'fold': 1, 'train_hasta': '2016-05-11', 'val_hasta': '2016-05-16', 'n_train': 36275, 'n_val': 12606}
{'fold': 2, 'train_hasta': '2016-05-16', 'val_hasta': '2016-05-19', 'n_train': 48881, 'n_val': 12556}
{'fold': 3, 'train_hasta': '2016-05-19', 'val_hasta': '2016-05-24', 'n_train': 61437, 'n_val': 7583}


## 5. Búsqueda de hiperparámetros (grid 3×3 + early stopping)

Grid 3×3 sobre `max_depth ∈ {4,6,8}` × `learning_rate ∈ {0.05, 0.1, 0.2}` = 9 combos. Otros hiperparámetros se fijan en valores estándar (`subsample=0.8`, `colsample_bytree=0.8`).

**`n_estimators` NO se tunea como dimensión:** se fija un tope de 1000 y se deja que `early_stopping_rounds=50` (sobre la val de cada fold) elija el número óptimo de árboles por combo. El `best_iteration` resultante se reporta como output derivado, no como hiperparámetro tuneado.

**`scale_pos_weight = neg/pos`:** compensa el desequilibrio (~80/20) durante el entrenamiento para mejorar la capacidad de discriminación (AUC-ROC, PR-AUC). La distorsión que esto introduce en la escala de probabilidad bruta se corregirá con la calibración aguas abajo — ver sección 7.

In [6]:
tabla_grid = modelado.buscar_hiperparametros(
    X_xgb, y_xgb, folds,
    grid_max_depth=modelado.GRID_MAX_DEPTH,
    grid_learning_rate=modelado.GRID_LEARNING_RATE,
    seed=SEED,
)
print("\nResultados del grid (ordenados por AUC media):")
display(tabla_grid)

INFO scale_pos_weight = neg/pos = 54458/14562 = 3.740


INFO Combo md=4 lr=0.05 → AUC=0.7459±0.0139, best_iter≈87


INFO Combo md=4 lr=0.10 → AUC=0.7461±0.0138, best_iter≈53


INFO Combo md=4 lr=0.20 → AUC=0.7455±0.0137, best_iter≈29


INFO Combo md=6 lr=0.05 → AUC=0.7443±0.0135, best_iter≈75


...

INFO Combo md=8 lr=0.05 → AUC=0.7394±0.0148, best_iter≈39


INFO Combo md=8 lr=0.10 → AUC=0.7393±0.0149, best_iter≈19


INFO Combo md=8 lr=0.20 → AUC=0.7354±0.0151, best_iter≈11



Resultados del grid (ordenados por AUC media):


,max_depth,learning_rate,auc_mean,auc_std,best_iteration_mean,auc_fold1,auc_fold2,auc_fold3,best_iter_fold1,best_iter_fold2,best_iter_fold3
0,4,0.10,0.746128,0.013827,52.666667,0.744998,0.729787,0.763599,49,42,67
1,4,0.05,0.745857,0.013867,86.666667,0.744461,0.729614,0.763496,138,66,56
2,4,0.20,0.745549,0.013727,28.666667,0.743214,0.730027,0.763407,32,37,17
3,6,0.05,0.744284,0.013471,75.000000,0.742963,0.728485,0.761404,94,72,59
4,6,0.10,0.743113,0.014919,33.333333,0.739882,0.726673,0.762785,54,27,19
5,6,0.20,0.741868,0.013766,12.000000,0.737614,0.727544,0.760448,15,13,8
6,8,0.05,0.739392,0.014764,38.666667,0.735025,0.723893,0.759258,31,75,10
7,8,0.10,0.739285,0.014899,19.000000,0.735071,0.723513,0.759271,19,27,11
8,8,0.20,0.735364,0.015117,11.000000,0.730639,0.719670,0.755783,19,7,7


## 6. Ajuste del modelo final en `train_xgb` con early stopping sobre `train_calib`

In [7]:
mejores = tabla_grid.iloc[0].to_dict()
print(f"Combo elegido: max_depth={int(mejores['max_depth'])}, learning_rate={mejores['learning_rate']}")
print(f"  AUC CV media: {mejores['auc_mean']:.4f} ± {mejores['auc_std']:.4f}")
print(f"  best_iteration medio CV: {mejores['best_iteration_mean']:.0f}\n")

modelo = modelado.ajustar_modelo_final(X_xgb, y_xgb, X_calib, y_calib, mejores, seed=SEED)
print(f"\nModelo final: max_depth={modelo.max_depth}, learning_rate={modelo.learning_rate}, best_iteration={modelo.best_iteration}")

INFO scale_pos_weight = neg/pos = 54458/14562 = 3.740


Combo elegido: max_depth=4, learning_rate=0.1
  AUC CV media: 0.7461 ± 0.0138
  best_iteration medio CV: 53



INFO Modelo final ajustado (md=4, lr=0.10, best_iteration=50).



Modelo final: max_depth=4, learning_rate=0.1, best_iteration=50


## 7. Calibración (Platt + Isotonic) sobre `train_calib`

Se ajustan ambos calibradores sobre el modelo congelado (`FrozenEstimator` — el XGBoost no se reentrena). La selección se hace por menor Brier sobre `train_calib`:

- Si `|Brier_iso - Brier_platt| < 0.001` → default Platt (más simple, menos riesgo de overfitting).
- En otro caso → el de menor Brier.

**Importante:** la calibración se aplica a las salidas del modelo ENTRENADO CON `scale_pos_weight`, no a una versión sin ponderar — la distorsión que corregimos es justamente la que introduce esa ponderación. La defensa en una frase es: *el modelo se optimiza primero para discriminación y se corrige después para precisión de probabilidad*.

In [8]:
calibrador, tabla_calibradores = modelado.ajustar_calibradores(modelo, X_calib, y_calib)
print(f"Calibrador elegido: {tabla_calibradores.attrs['metodo_elegido']}")
print(f"Motivo: {tabla_calibradores.attrs['motivo_seleccion']}\n")
display(tabla_calibradores)

INFO Brier en train_calib: raw=0.22137, Platt=0.13787, Isotonic=0.13685. Elegido: isotonic (Isotonic mejor por -0.00103).


Calibrador elegido: isotonic
Motivo: Isotonic mejor por -0.00103



,metodo,brier_train_calib,seleccionado
0,sin_calibrar,0.221373,False
1,platt_sigmoid,0.137872,False
2,isotonic,0.136846,True


## 8. Evaluación en test (una sola vez)

`prob_noshow_calibrada` es la salida que alimenta NB05 (segmentación) y NB06 (Monte Carlo).

In [9]:
metricas, prob_raw, prob_cal = modelado.calcular_metricas_test(modelo, calibrador, X_test, y_test)
print(f"\nMétricas en test:")
print(f"  AUC-ROC:           {metricas.auc_roc:.4f}")
print(f"  PR-AUC:            {metricas.pr_auc:.4f}")
print(f"  Brier raw:         {metricas.brier_raw:.4f}")
print(f"  Brier calibrado:   {metricas.brier_calibrado:.4f}  (Δ={metricas.brier_raw - metricas.brier_calibrado:+.4f})")
print(f"  Prevalencia test:  {metricas.prevalencia:.4f}")

INFO Test: AUC=0.7302, PR-AUC=0.3286, Brier raw=0.2165, Brier calibrado=0.1367, prev=0.186



Métricas en test:
  AUC-ROC:           0.7302
  PR-AUC:            0.3286
  Brier raw:         0.2165
  Brier calibrado:   0.1367  (Δ=+0.0798)
  Prevalencia test:  0.1865


## 9. Matriz de confusión a tres umbrales

Tres umbrales reportados en una sola tabla:

1. **`top20` (operacional, headline)**: umbral = percentil 80 de la probabilidad calibrada en test. Corresponde a un presupuesto realista de SMS sobre el 20% del volumen mensual.
2. **`literatura_0.5`**: clasificación nominal. Se incluye como referencia, anotando que con prevalencia ~20% rara vez se cruza.
3. **`prevalencia` (informado por prevalencia)**: umbral = tasa observada de no-show en test. NO se etiqueta como "Youden-like" porque no se optimiza J explícitamente; es simplemente el umbral natural cuando se quiere equilibrar precision/recall a la tasa base.

In [10]:
tabla_confusion = modelado.matriz_confusion_tres_umbrales(y_test, prob_cal)
display(tabla_confusion)

INFO Matriz de confusión calculada a 3 umbrales: top20=0.2985, 0.5, prev=0.1865


,umbral_nombre,umbral_valor,TN,FP,FN,TP,precision,recall,n_positivos_predichos,pct_positivos_predichos
0,top20,0.298546,14365,2988,2379,1598,0.348452,0.401810,4586,0.215002
1,literatura_0.5,0.500000,17334,19,3942,35,0.648148,0.008801,54,0.002532
2,prevalencia,0.186451,8037,9316,458,3519,0.274172,0.884838,12835,0.601735


## 10. Curva de calibración (test) + sanity check interno

**Diagrama de fiabilidad (test):** probabilidad media predicha vs frecuencia observada por bin (cuantil).

Lectura geométrica:
- Curva **por debajo** de la diagonal ⇒ el modelo **sobreestima** el riesgo (predicción > observación).
- Curva **por encima** ⇒ el modelo **subestima** el riesgo (predicción < observación).
- Curva pegada a la diagonal ⇒ probabilidades bien calibradas.

La narrativa esperada: la curva raw sobrestima la probabilidad de no-show (efecto de `scale_pos_weight`); tras calibrar, las probabilidades se alinean con las frecuencias observadas — y la caída del Brier en test cuantifica esa mejora.

**Sanity check (lectura cauta):** se reproduce la curva sobre un slice interno del periodo de entrenamiento, justo antes de `train_calib`. Para que la comparación sea like-with-like se usa `strategy="uniform"` (bin edges idénticas en ambas líneas). Si las dos curvas son visualmente similares en su rango común, la calibración es estable; si divergen, se reporta como variabilidad entre submuestras — no como prueba contundente de robustez ni como fallo crítico.

In [11]:
modelado.plot_curva_calibracion(
    y_test, prob_raw, prob_cal,
    ruta_salida=rutas.FIGURAS / "nb03_calibracion_curva_v1.png",
)

# Sanity check: slice interno justo antes del corte train_xgb/train_calib
fechas_xgb = train_df["AppointmentDay"].loc[X_xgb.index]
corte_interno = fechas_xgb.quantile(0.85, interpolation="lower")
mask_interno = fechas_xgb > corte_interno
X_interno = X_xgb.loc[mask_interno]
y_interno = y_xgb.loc[mask_interno]
prob_interno = calibrador.predict_proba(X_interno)[:, 1]
print(f"Slice interno (sanity): {mask_interno.sum()} filas, fechas > {pd.Timestamp(corte_interno).date()}")

modelado.plot_curva_calibracion_sanity(
    y_interno, prob_interno, y_test, prob_cal,
    ruta_salida=rutas.FIGURAS / "nb03_calibracion_sanity_v1.png",
)

INFO Curva de calibración guardada en /Users/sergi/Library/CloudStorage/OneDrive-Personal/Documentos/TFG Analytics Scripts/Kaggle Processor/outputs/figuras/nb03_calibracion_curva_v1.png


Slice interno (sanity): 7583 filas, fechas > 2016-05-19


INFO Sanity check guardado en /Users/sergi/Library/CloudStorage/OneDrive-Personal/Documentos/TFG Analytics Scripts/Kaggle Processor/outputs/figuras/nb03_calibracion_sanity_v1.png (slice: 5 bins válidos, test: 5)


PosixPath('/Users/sergi/Library/CloudStorage/OneDrive-Personal/Documentos/TFG Analytics Scripts/Kaggle Processor/outputs/figuras/nb03_calibracion_sanity_v1.png')

## 11. SHAP — interpretabilidad

**Compute:** TreeExplainer se ejecuta sobre TODO el test set (~21k filas) — coste ~20–60s en M5 Pro. Los plots usan una submuestra estratificada de 5,000 filas (por cuartil de riesgo predicho × outcome real), porque por encima de ~5k puntos los beeswarms y dependence plots se vuelven ilegibles. **El cálculo de métricas globales (ranking de features) usa los SHAP values completos**, no la submuestra.

**Interpretación enmarcada:** `is_first_visit` y `prior_noshow_rate` codifican la misma señal cuando la primera vale 1 (censura por la izquierda → no hay historial observado para esa fila). SHAP repartirá atribución entre ambas — no es un bug.

**Nota sobre `barrio (frecuencia)`:** la variable es `neighbourhood_encoded`, una codificación por frecuencia del barrio (proporción de citas en train que pertenecen a ese barrio). No es el identificador del barrio: valores altos = barrios con muchas citas en el dataset; valores bajos = barrios poco representados. La codificación por frecuencia se eligió por encima de target-encoding, que filtraría `Showed_up` al modelo.

In [12]:
shap_values = modelado.calcular_shap_values_test(modelo, X_test)
idx_muestra = modelado.submuestra_estratificada_shap(X_test, y_test, prob_cal, n=modelado.N_SHAP_VISUALIZACION, seed=SEED)

importancia_global = pd.Series(
    np.abs(shap_values.values).mean(axis=0),
    index=X_test.columns,
).sort_values(ascending=False)
print("Top 10 features por |SHAP| media (test completo):")
print(importancia_global.head(10).to_string())

INFO SHAP values calculados sobre test: (21330, 18)


INFO Submuestra SHAP estratificada: 5001 filas en 8 estratos.


Top 10 features por |SHAP| media (test completo):
lead_time                  0.844444
Age                        0.223533
prior_noshow_rate          0.082186
neighbourhood_encoded      0.058655
scheduled_month            0.028146
prior_appointment_count    0.025822
Scholarship                0.023216
gender_F                   0.022666
appointment_weekday        0.015109
scheduled_weekday          0.009121


In [13]:
modelado.plot_shap_summary(
    shap_values, X_test, idx_muestra,
    ruta_salida=rutas.FIGURAS / "nb03_shap_beeswarm_v1.png",
)

top4 = [c for c in importancia_global.head(8).index if X_test[c].nunique() > 5][:4]
print(f"Dependence plots para: {top4}")
modelado.plot_shap_dependence(
    shap_values, X_test, idx_muestra, features=top4,
    ruta_salida=rutas.FIGURAS / "nb03_shap_dependence_v1.png",
)

INFO SHAP summary guardado en /Users/sergi/Library/CloudStorage/OneDrive-Personal/Documentos/TFG Analytics Scripts/Kaggle Processor/outputs/figuras/nb03_shap_beeswarm_v1.png


INFO SHAP dependence guardado en /Users/sergi/Library/CloudStorage/OneDrive-Personal/Documentos/TFG Analytics Scripts/Kaggle Processor/outputs/figuras/nb03_shap_dependence_v1.png


Dependence plots para: ['lead_time', 'Age', 'prior_noshow_rate', 'neighbourhood_encoded']


PosixPath('/Users/sergi/Library/CloudStorage/OneDrive-Personal/Documentos/TFG Analytics Scripts/Kaggle Processor/outputs/figuras/nb03_shap_dependence_v1.png')

### Explicaciones locales SHAP — tres casos (PNGs separados)

Cada arquetipo se renderiza como figura independiente (un PNG por caso) para que las etiquetas de features no se solapen entre paneles. Cada figura anota la **probabilidad calibrada de no-show** del caso, junto al `f(x)` en log-odds que SHAP muestra por defecto.

- **Caso 1**: alto riesgo predicho y no-show observado (priorización correcta).
- **Caso 2**: alto riesgo predicho pero asistió (falso positivo del punto operativo).
- **Caso 3**: bajo riesgo predicho con no-show observado (falso negativo).

Para el documento principal del TFG basta con incluir uno (típicamente el Caso 1). Los otros dos quedan disponibles como anexo si se quiere ilustrar los dos tipos de error.

In [14]:
casos = modelado.elegir_casos_locales(y_test, prob_cal, seed=SEED)
print(f"Casos locales (iloc): {casos}")

rutas_waterfall = modelado.plot_shap_local_waterfall_individuales(
    shap_values, X_test, casos, prob_cal=prob_cal,
    directorio_salida=rutas.FIGURAS,
    prefijo="nb03_shap_local_waterfall",
)
for clave, ruta in rutas_waterfall.items():
    print(f"  {clave}: {ruta.name} (prob_cal = {prob_cal[casos[clave]]:.3f})")

INFO Casos locales SHAP: {'caso1_alto_riesgo_y_noshow': 2342, 'caso2_alto_riesgo_sin_noshow': 20607, 'caso3_bajo_riesgo_con_noshow': 12390}


Casos locales (iloc): {'caso1_alto_riesgo_y_noshow': 2342, 'caso2_alto_riesgo_sin_noshow': 20607, 'caso3_bajo_riesgo_con_noshow': 12390}


INFO SHAP local caso1_alto_riesgo_y_noshow guardado en /Users/sergi/Library/CloudStorage/OneDrive-Personal/Documentos/TFG Analytics Scripts/Kaggle Processor/outputs/figuras/nb03_shap_local_waterfall_caso1_alto_riesgo_y_noshow_v1.png (prob_cal=0.354)


INFO SHAP local caso2_alto_riesgo_sin_noshow guardado en /Users/sergi/Library/CloudStorage/OneDrive-Personal/Documentos/TFG Analytics Scripts/Kaggle Processor/outputs/figuras/nb03_shap_local_waterfall_caso2_alto_riesgo_sin_noshow_v1.png (prob_cal=0.354)


INFO SHAP local caso3_bajo_riesgo_con_noshow guardado en /Users/sergi/Library/CloudStorage/OneDrive-Personal/Documentos/TFG Analytics Scripts/Kaggle Processor/outputs/figuras/nb03_shap_local_waterfall_caso3_bajo_riesgo_con_noshow_v1.png (prob_cal=0.025)


  caso1_alto_riesgo_y_noshow: nb03_shap_local_waterfall_caso1_alto_riesgo_y_noshow_v1.png (prob_cal = 0.354)
  caso2_alto_riesgo_sin_noshow: nb03_shap_local_waterfall_caso2_alto_riesgo_sin_noshow_v1.png (prob_cal = 0.354)
  caso3_bajo_riesgo_con_noshow: nb03_shap_local_waterfall_caso3_bajo_riesgo_con_noshow_v1.png (prob_cal = 0.025)


## 12. Persistencia de artefactos

- **Modelo XGBoost** en formato JSON nativo (`outputs/modelos/xgboost_v1.json`).
- **Calibrador** como pickle (`outputs/modelos/calibrador_v1.pkl`).
- **Probabilidades de no-show** sobre el test set (`outputs/reportes/noshow_probs_v1.csv`) — entrada de NB05 y NB06.
- **Sidecar de metadatos** (`outputs/reportes/nb03_metadatos_v1.json`).

In [15]:
ruta_modelo = modelado.guardar_modelo(modelo, rutas.MODELOS / "xgboost_v1.json")
ruta_calib = modelado.guardar_calibrador(calibrador, rutas.MODELOS / "calibrador_v1.pkl")
ruta_probs = modelado.exportar_probabilidades_test(
    test_df, prob_cal=prob_cal, prob_raw=prob_raw,
    ruta_csv=rutas.REPORTES / "noshow_probs_v1.csv",
)

umbral_top20 = float(tabla_confusion.loc[tabla_confusion["umbral_nombre"] == "top20", "umbral_valor"].iloc[0])
spw_train = (y_train_full == 0).sum() / (y_train_full == 1).sum()

metadatos = modelado.escribir_metadatos_nb03(
    rutas.REPORTES / "nb03_metadatos_v1.json",
    seed=SEED,
    n_train=len(train_df),
    n_train_xgb=len(X_xgb),
    n_train_calib=len(X_calib),
    n_test=len(test_df),
    fecha_corte_xgb_calib=fecha_corte_xgb_calib,
    folds=folds,
    tabla_grid=tabla_grid,
    mejores_params=mejores,
    tabla_calibradores=tabla_calibradores,
    metricas_test=metricas,
    tabla_confusion=tabla_confusion,
    umbral_top20=umbral_top20,
    features_usadas=modelado.FEATURES_NB03,
    scale_pos_weight=float(spw_train),
    casos_locales=casos,
)
print(f"\nArtefactos guardados:")
print(f"  {ruta_modelo}")
print(f"  {ruta_calib}")
print(f"  {ruta_probs}")
print(f"  {rutas.REPORTES / 'nb03_metadatos_v1.json'}")

INFO Modelo XGBoost guardado en /Users/sergi/Library/CloudStorage/OneDrive-Personal/Documentos/TFG Analytics Scripts/Kaggle Processor/outputs/modelos/xgboost_v1.json


INFO Calibrador guardado en /Users/sergi/Library/CloudStorage/OneDrive-Personal/Documentos/TFG Analytics Scripts/Kaggle Processor/outputs/modelos/calibrador_v1.pkl


INFO Probabilidades exportadas a /Users/sergi/Library/CloudStorage/OneDrive-Personal/Documentos/TFG Analytics Scripts/Kaggle Processor/outputs/reportes/noshow_probs_v1.csv (21330 filas)


INFO Metadatos NB03 guardados en /Users/sergi/Library/CloudStorage/OneDrive-Personal/Documentos/TFG Analytics Scripts/Kaggle Processor/outputs/reportes/nb03_metadatos_v1.json



Artefactos guardados:
  /Users/sergi/Library/CloudStorage/OneDrive-Personal/Documentos/TFG Analytics Scripts/Kaggle Processor/outputs/modelos/xgboost_v1.json
  /Users/sergi/Library/CloudStorage/OneDrive-Personal/Documentos/TFG Analytics Scripts/Kaggle Processor/outputs/modelos/calibrador_v1.pkl
  /Users/sergi/Library/CloudStorage/OneDrive-Personal/Documentos/TFG Analytics Scripts/Kaggle Processor/outputs/reportes/noshow_probs_v1.csv
  /Users/sergi/Library/CloudStorage/OneDrive-Personal/Documentos/TFG Analytics Scripts/Kaggle Processor/outputs/reportes/nb03_metadatos_v1.json


## 13. Checklist auto-verificable

In [16]:
assert "SMS_received" not in modelado.FEATURES_NB03, "SMS_received NO debe estar en el modelo de baseline."
assert "Showed_up" not in modelado.FEATURES_NB03, "Showed_up es el target, no feature."
assert metricas.auc_roc > 0.6, f"AUC sospechosamente baja: {metricas.auc_roc}"
assert metricas.brier_calibrado <= metricas.brier_raw + 0.01, "La calibración no debería empeorar Brier sustancialmente."
assert (prob_cal >= 0).all() and (prob_cal <= 1).all(), "Probabilidades calibradas fuera de [0, 1]."
assert ruta_modelo.exists() and ruta_calib.exists() and ruta_probs.exists(), "Artefactos persistidos faltantes."
for clave, ruta in rutas_waterfall.items():
    assert ruta.exists(), f"Waterfall faltante: {ruta}"

print("NB03 completado: modelo, calibrador y probabilidades persistidos.")
print(f"   AUC test = {metricas.auc_roc:.4f} | Brier calibrado = {metricas.brier_calibrado:.4f}")
print(f"   Waterfalls: {len(rutas_waterfall)} PNGs independientes.")
print(f"   Siguiente paso: NB04 (inferencia causal IPW).")

NB03 completado: modelo, calibrador y probabilidades persistidos.
   AUC test = 0.7302 | Brier calibrado = 0.1367
   Waterfalls: 3 PNGs independientes.
   Siguiente paso: NB04 (inferencia causal IPW).
